In [87]:
import os
import random
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from collections import Counter
from torch.nn.utils.rnn import pad_sequence
import csv

# =========================
# CONFIG
# =========================
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

DATA_DIR = Path('./data/processed/')
TRAIN_IMG_DIR = DATA_DIR / 'train'
VAL_IMG_DIR   = DATA_DIR / 'val'
TEST_IMG_DIR  = DATA_DIR / 'test'

TRAIN_CAPTIONS_CSV = DATA_DIR / 'train_captions.csv'
VAL_CAPTIONS_CSV   = DATA_DIR / 'val_captions.csv'
TEST_CAPTIONS_CSV  = DATA_DIR / 'test_captions.csv'

EMBED_SIZE = 256
HIDDEN_SIZE = 512
NUM_LAYERS = 1
BATCH_SIZE = 128
LR = 1e-3
NUM_EPOCHS = 10
MIN_WORD_FREQ = 5
MAX_LEN = 30

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

Device: cuda:0


In [88]:
# =========================
# Fonctions utilitaires
# =========================

def clean_caption(c):
    import re
    c = c.lower()
    c = c.replace("'s", "")
    c = re.sub(r"[^a-z0-9\s]", '', c)
    c = re.sub(r"\s+", ' ', c).strip()
    return c

def load_captions(csv_path):
    captions_by_image = {}
    if not csv_path.exists():
        raise FileNotFoundError(f"Captions CSV not found at {csv_path}")
    
    with open(csv_path, newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f, delimiter='|')  # Important: '|'
        expected_cols = {'image', 'caption'}
        if not expected_cols.issubset(reader.fieldnames):
            raise ValueError(f"CSV must contain columns: image|caption. Found: {reader.fieldnames}")
        for row in reader:
            img = row['image']
            cap = clean_caption(row['caption'])
            cap = '<start> ' + cap + ' <end>'
            captions_by_image.setdefault(img, []).append(cap)
    return captions_by_image


In [89]:
# =========================
# Charger captions et construire vocab
# =========================
train_captions = load_captions(TRAIN_CAPTIONS_CSV)
val_captions   = load_captions(VAL_CAPTIONS_CSV)
test_captions  = load_captions(TEST_CAPTIONS_CSV)

counter = Counter()
for caps in train_captions.values():
    for c in caps:
        counter.update(c.split())

words = [w for w, cnt in counter.items() if cnt >= MIN_WORD_FREQ]
itos = ['<pad>', '<unk>'] + words
stoi = {w:i for i,w in enumerate(itos)}
vocab_size = len(itos)
print("Vocab size:", vocab_size)

def numericalize(token_list):
    return [stoi.get(t, stoi['<unk>']) for t in token_list]

Vocab size: 2654


In [90]:
# =========================
# Dataset pour .pt pré-traités
# =========================
class Flickr8kDataset(Dataset):
    def __init__(self, img_dir, captions_dict, max_len=MAX_LEN):
        self.img_dir = Path(img_dir)
        self.max_len = max_len
        self.entries = []
        for img_name, caps in captions_dict.items():
            img_path = self.img_dir / (img_name.replace('.jpg', '.pt'))
            if img_path.exists():
                for c in caps:
                    tokens = c.split()
                    if len(tokens) <= max_len:
                        self.entries.append((str(img_path), tokens))
            else:
                print("File not found:", img_path)  # debug

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, idx):
        img_path, tokens = self.entries[idx]
        image_tensor = torch.load(img_path)  # tensor pré-traité
        ids = numericalize(tokens)
        return image_tensor, torch.tensor(ids, dtype=torch.long)

In [91]:
# =========================
# DataLoader + collate_fn
# =========================
def collate_fn(batch):
    imgs, caps = zip(*batch)
    imgs = torch.stack(imgs)
    lengths = torch.tensor([len(c) for c in caps], dtype=torch.long)
    caps_padded = pad_sequence(caps, batch_first=True, padding_value=stoi['<pad>'])
    return imgs, caps_padded, lengths

train_ds = Flickr8kDataset(TRAIN_IMG_DIR, train_captions)
val_ds   = Flickr8kDataset(VAL_IMG_DIR, val_captions)
test_ds  = Flickr8kDataset(TEST_IMG_DIR, test_captions)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, pin_memory=True, num_workers=4)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, pin_memory=True, num_workers=4)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, pin_memory=True, num_workers=4)

print("Train dataset size:", len(train_ds))

Train dataset size: 32335


In [92]:
# =========================
# Encoder CNN (features)
# =========================
import torchvision.models as models

class EncoderCNN(nn.Module):
    def __init__(self, embed_size, train_cnn=False):
        super().__init__()
        resnet = models.resnet18(pretrained=True)
        modules = list(resnet.children())[:-1]  # remove fc
        self.resnet = nn.Sequential(*modules)
        for p in self.resnet.parameters():
            p.requires_grad = train_cnn
        self.fc = nn.Linear(resnet.fc.in_features, embed_size)
        self.bn = nn.BatchNorm1d(embed_size, momentum=0.01)

    def forward(self, images):
        with torch.set_grad_enabled(self.fc.weight.requires_grad):
            features = self.resnet(images).squeeze()
        features = self.fc(features)
        features = self.bn(features)
        return features


In [93]:
import torch.nn.functional as F

class MyCNN(nn.Module):
    def __init__(self, embed_size):
        super().__init__()
        # Convolutions simples
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.pool = nn.MaxPool2d(2, 2)
        
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        
        # Flatten + fully connected
        self.fc = nn.Linear(128 * 28 * 28, embed_size)  # si input 224x224

        self.embed_size = embed_size

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))
        x = x.view(x.size(0), -1)  # flatten
        x = self.fc(x)
        return x


In [95]:
# =========================
# Decoder LSTM corrigé
# =========================
class DecoderLSTM(nn.Module):
    def __init__(self, embed_size, hidden_size, vocab_size, num_layers=1, dropout=0.5):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_size)
        self.lstm = nn.LSTM(embed_size, hidden_size, num_layers=num_layers, batch_first=True, dropout=dropout)
        self.linear = nn.Linear(hidden_size, vocab_size)
        self.init_h = nn.Linear(embed_size, hidden_size)
        self.init_c = nn.Linear(embed_size, hidden_size)

    def forward(self, features, captions, lengths):
        embeddings = self.embed(captions)
        features = features.unsqueeze(1)
        embeddings = torch.cat((features, embeddings[:, :-1, :]), dim=1)
        lengths = lengths.cpu()  # must be CPU for pack_padded_sequence
        packed = pack_padded_sequence(embeddings, lengths, batch_first=True, enforce_sorted=False)
        hiddens, _ = self.lstm(packed)
        outputs = self.linear(hiddens[0])
        return outputs

    def sample(self, features, max_len=MAX_LEN):
        sampled_ids = []
        inputs = features.unsqueeze(1)
        states = (self.init_h(features).unsqueeze(0), self.init_c(features).unsqueeze(0))
        for i in range(max_len):
            hiddens, states = self.lstm(inputs, states)
            outputs = self.linear(hiddens.squeeze(1))
            _, predicted = outputs.max(1)
            sampled_ids.append(predicted)
            inputs = self.embed(predicted).unsqueeze(1)
        sampled_ids = torch.stack(sampled_ids, 1)
        return sampled_ids

In [96]:
# =========================
# Initialiser modèles, loss, optim
# =========================
encoder = EncoderCNN(EMBED_SIZE).to(DEVICE)
# encode = MyCNN(EMBED_SIZE).to(DEVICE)
decoder = DecoderLSTM(EMBED_SIZE, HIDDEN_SIZE, vocab_size, NUM_LAYERS).to(DEVICE)

criterion = nn.CrossEntropyLoss(ignore_index=stoi['<pad>'])
params = list(decoder.parameters()) + list(encoder.fc.parameters()) + list(encoder.bn.parameters())
optimizer = torch.optim.Adam(params, lr=LR)

In [97]:

# =========================
# Fonctions entraînement / validation
# =========================
def train_one_epoch(epoch):
    encoder.train()
    decoder.train()
    running_loss = 0.0
    for i, (images, captions, lengths) in enumerate(train_loader):
        images, captions, lengths = images.to(DEVICE), captions.to(DEVICE), lengths.to(DEVICE)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            features = encoder(images)
            outputs = decoder(features, captions, lengths)
            # pack targets (skip <start>)
            targets = pack_padded_sequence(captions[:, 1:], (lengths-1).cpu(), batch_first=True, enforce_sorted=False).data
            loss = criterion(outputs, targets)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item()
        if (i+1) % 100 == 0:
            print(f"Epoch [{epoch}], Step [{i+1}/{len(train_loader)}], Loss: {running_loss/(i+1):.4f}")
    return running_loss / len(train_loader)


# =========================
def validate():
    encoder.eval()
    decoder.eval()
    total_loss = 0.0
    with torch.no_grad():
        for images, captions, lengths in val_loader:
            images, captions, lengths = images.to(DEVICE), captions.to(DEVICE), lengths.to(DEVICE)
            with torch.cuda.amp.autocast():
                features = encoder(images)
                outputs = decoder(features, captions, lengths)
                targets = pack_padded_sequence(captions[:,1:], (lengths-1).cpu(), batch_first=True, enforce_sorted=False).data
                loss = criterion(outputs, targets)
            total_loss += loss.item()
    return total_loss / len(val_loader)


In [98]:
best_val_loss = float('inf')
for epoch in range(1, NUM_EPOCHS+1):
    train_loss = train_one_epoch(epoch)
    val_loss = validate()
    print(f"Epoch {epoch} finished. Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            'encoder': encoder.state_dict(),
            'decoder': decoder.state_dict(),
            'stoi': stoi,
            'itos': itos
        }, 'fromSCRmodel.pth')
        print("Saved best model.")

/tmp/ipykernel_1030766/3009809162.py:11: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


ValueError: Expected input batch_size (1656) to match target batch_size (1528).

In [4]:
# =========================
# Génération de captions (greedy)
# =========================
def decode_ids(id_list):
    return [itos[i] if i < len(itos) else '<unk>' for i in id_list]

def generate_caption(image_tensor, max_len=MAX_LEN):
    encoder.eval()
    decoder.eval()
    with torch.no_grad():
        if image_tensor.dim() == 3:  # (C,H,W) → ajouter batch
            image_tensor = image_tensor.unsqueeze(0)
        image_tensor = image_tensor.to(DEVICE)
        features = encoder(image_tensor)
        sampled_ids = decoder.sample(features, max_len=max_len)
        sampled_ids = sampled_ids[0].cpu().numpy().tolist()
        
        tokens = []
        for idx in sampled_ids:
            tok = itos[idx] if idx < len(itos) else '<unk>'
            if tok == '<end>':
                break
            tokens.append(tok)
        if tokens and tokens[0] == '<start>':
            tokens = tokens[1:]
        return ' '.join(tokens)

In [5]:
# =========================
# Évaluation BLEU sur le test set
# =========================
import nltk
from nltk.translate.bleu_score import corpus_bleu
# nltk.download('punkt')  # si pas déjà téléchargé

references = []
hypotheses = []

for images, caps, lengths in test_loader:
    for i in range(images.size(0)):
        img = images[i]
        gen_caption = generate_caption(img)
        
        # reference: toutes les captions pour cette image
        img_path, _ = test_ds.entries[i]
        img_name = Path(img_path).name.replace('.pt', '.jpg')
        refs = test_captions.get(img_name, [])
        if not refs:
            continue
        ref_tok = [r.replace('<start> ','').replace(' <end>','').split() for r in refs]
        hyp_tok = gen_caption.split()
        references.append(ref_tok)
        hypotheses.append(hyp_tok)

if references and hypotheses:
    bleu1 = corpus_bleu(references, hypotheses, weights=(1,0,0,0))
    bleu4 = corpus_bleu(references, hypotheses, weights=(0.25,0.25,0.25,0.25))
    print("BLEU-1:", bleu1)
    print("BLEU-4:", bleu4)
else:
    print("No hypotheses or references for BLEU evaluation.")

ValueError: expected 2D or 3D input (got 1D input)

In [ ]:
# Afficher N exemples du test set
N = 5
encoder.eval()
decoder.eval()

for i in range(N):
    # choisir image et caption du test_ds
    img_tensor, true_caption_ids = test_ds[i]
    
    # générer caption
    gen_caption = generate_caption(img_tensor)
    
    # true caption (prendre la première référence)
    true_caption = decode_ids(true_caption_ids.tolist())
    # enlever <start> et <end>
    if true_caption[0] == '<start>':
        true_caption = true_caption[1:]
    if '<end>' in true_caption:
        true_caption = true_caption[:true_caption.index('<end>')]
    true_caption = ' '.join(true_caption)
    
    print(f"Image {i+1}:")
    print("Generated caption:", gen_caption)
    print("True caption    :", true_caption)
    print("-"*50)
